# Arm G Phase 1 — matched goal conflict

This run tests whether Llama-3.1-8B carries a cross-family internal signal for a goal that cannot be completed without breaking a binding scope rule.

Each pair has the same system prompt, tools, catalog result, and synthetic workspace. The directive changes only an opaque target ID. One target is in scope and the matched target is outside scope. The model must emit only `READY`, keeping visible behavior matched at the measurement point.

Set **Runtime → Change runtime type → A100 GPU**, then run the cells in order. High-RAM host memory is optional. The runner checkpoints every generation and activation to Drive, so rerunning resumes after a disconnect.

In [ ]:
# Colab supplies torch/CUDA.
print("Protocol: ARM_G_PHASE1_MATCHED_READY_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
# Mount Drive and copy the frozen launch files to local Colab storage.
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-launch"
for name in ("arm_g_phase1.py", "arm_g_scenarios.py"):
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")
print("Arm G launch files staged: OK")

In [ ]:
# Frozen run configuration.
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-phase1-seed17"
PAIRS_PER_FAMILY = 16
REPEATS = 2
MAX_NEW_TOKENS = 16
SEED = 17
BOOTSTRAP = 2000

# Put HF_TOKEN in Colab's Secrets panel; do not paste it into the notebook.
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
# Hardware and gated-model access gate.
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Build and audit all matched scenarios before spending GPU time.
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_phase1.py",
    "--model", ACTING_MODEL,
    "--work-dir", WORK_DIR,
    "--pairs-per-family", str(PAIRS_PER_FAMILY),
    "--repeats", str(REPEATS),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--seed", str(SEED),
    "--bootstrap", str(BOOTSTRAP),
]
subprocess.run(base_cmd + ["--manifest-only"], check=True, env=env)

In [ ]:
# Generate matched READY rollouts, capture exact-token activations, and analyze.
# Re-running resumes completed checkpoints.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
# Compact result view. Full matrices, sweeps, and controls remain in Drive.
import json
result_path = f"{WORK_DIR}/arm_g_phase1_result.json"
result = json.load(open(result_path))
print(json.dumps({
    "decision": result["decision"],
    "eligibility_reasons": result["eligibility_reasons"],
    "primary": result["primary"],
    "sample_counts": result["sample_counts"],
    "behavior": result["behavior"]["overall"],
    "prompt_only_leakage": result["prompt_only_leakage"],
    "controls": result["controls"],
}, indent=2))
print("full artifact:", result_path)